### 데이터로드 및 패키지로드

In [1]:
# -------------------------------
# 사용자 정의 모듈
# -------------------------------
from hossam import *


# -------------------------------
# 데이터 처리 / 시각화
# -------------------------------
import pandas as pd
import numpy as np
import seaborn as sb
from pandas import DataFrame
from matplotlib import pyplot as plt


# -------------------------------
# 머신러닝 (회귀 / 전처리 / 검증)
# -------------------------------
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    learning_curve,
)


# -------------------------------
# 머신러닝 평가 지표
# -------------------------------
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
)


# -------------------------------
# 통계 검정 / 진단
# -------------------------------
from scipy.stats import (
    zscore,
    probplot,
    shapiro,
    kstest,
    jarque_bera,
)

from statsmodels.stats.api import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.graphics.gofplots import qqplot as sm_qqplot
import statsmodels.api as sm


In [2]:
origin=load_data("fish_processed")
origin.head()

농어의 길이,높이,두께,무게를 조사한 데이터의 전처리 버전


,길이,높이,두께,무게
0,-2.180225,-2.016507,-1.896175,1.931521
1,-1.587434,-1.518703,-1.560774,3.496508
2,-1.442032,-1.417039,-1.316328,3.713572
3,-1.307815,-1.147103,-1.202633,3.960813
4,-1.173599,-1.147103,-1.026405,4.262680


### 농어를 탐색해봐요

## 바닷속을 알아보자

# -----------------------------------------

### 바닷속 파헤쳐보기
- **길이**
- **높이**
- **두께**
- **무게**

### 표
| 농어 | ㅇㅇㅇㅇ | ㅇㅇㅇㅇ | ㅇㅇㅇㅇ | ㅇㅇㅇㅇ |
|------|---------|---------|---------|---------|
|  | - | - | - | - |
|  | - | - | - | - |
|  | - | - | - | - |

In [6]:
df = origin.copy()
# 데이터프레임에서 필요한 열만 선택
DF = df[['길이', '두께', '무게', '높이']]

DF.describe().T



,count,mean,std,min,25%,50%,75%,max
길이,56.0,1.586033e-17,1.009050,-2.180225,-0.678674,-0.290004,0.976667,1.801542
두께,56.0,-6.344132e-17,1.009050,-1.896175,-0.696689,-0.335706,0.968949,1.929674
무게,56.0,5.468641e+00,1.078597,1.931521,4.795791,5.338669,6.541732,7.003974
높이,56.0,2.061843e-16,1.009050,-2.016507,-0.761480,-0.330284,1.047442,1.731046


In [18]:
# 방법 1: 간단한 반복문
df_desc = df.describe()
max_values = []

for col in df_desc.columns:
    max_values.append(df_desc.loc['max', col])

print(max_values)





[np.float64(1.801541779923801), np.float64(1.731046275384641), np.float64(1.929674421262049), np.float64(7.00397413672268)]


In [21]:
import numpy as np
import pandas as pd

# 예시: 이미 있는 DataFrame `df`
# 1) 컬럼별로 리스트(및 np.float)로 변환
col_lists = {col: df[col].astype(float).tolist() for col in df.columns}

# 2) describe()에서 max 값 추출 (float으로)
max_series = df.describe().loc['mean'].astype(float)   # Series, index = column names
max_values_list = max_series.tolist()                 # 리스트 형태

print("컬럼별 리스트 예시:", list(col_lists.items())[:1])
print("max_values_list:", max_values_list)

# 3) 컬럼별로 max와 매칭해서 DataFrame 생성(각 행별 비교)
match_dfs = {}
for col in df.columns:
    col_float = df[col].astype(float)
    match_dfs[col] = pd.DataFrame({
        'value': col_float,
        'max': float(max_series[col]),
        'is_equal_to_max': col_float == max_series[col],
        'diff_from_max': max_series[col] - col_float
    })

# 특정 컬럼 결과 출력 예시
print(match_dfs[df.columns[0]].head())

# 4) 만약 각 컬럼의 리스트와 max_values_list를 순서대로 대응시켜 묶고 싶다면
paired = [(col, col_lists[col], float(max_series[col])) for col in df.columns]
# 또는 각 컬럼별 value 리스트와 max를 함께 보기
for col, vals, m in paired:
    print(f"\nColumn: {col}, max={m}")
    print("values (first5):", vals[:5])

컬럼별 리스트 예시: [('길이', [-2.180225063153903, -1.587433932021549, -1.442032333819273, -1.307815473940249, -1.173598614061226, -1.106490184121714, -1.02819701585895, -0.9946428008891938, -0.9275343709496817, -0.8827954176566739, -0.7709480344241542, -0.7709480344241542, -0.7709480344241542, -0.7373938194543982, -0.6591006511916344, -0.6591006511916344, -0.6591006511916344, -0.6591006511916344, -0.6591006511916344, -0.6031769595753745, -0.6031769595753745, -0.5808074829288706, -0.5472532679591146, -0.4913295763428547, -0.4354058847265948, -0.4354058847265948, -0.3682974547870828, -0.3235585014940751, -0.256450071554563, -0.1557874266452954, -0.06630952005927945, -0.04394004341277558, -0.04394004341277558, -0.04394004341277558, 0.01198364820348432, 0.09027681646624809, 0.2356784146685239, 0.5488510877195789, 0.7389916392148629, 0.7949153308311228, 0.9626864056799025, 0.9067627140636426, 1.018610097296162, 1.018610097296162, 1.242304863761202, 1.242304863761202, 1.242304863761202, 1.35415224699

In [ ]:

print(match_dfs[df.columns[0]].head())

p=[(col,col_lists[col],float(max_series[col])) for col in df.columns]

# 길이 높이 두께 무게
for col, vals, m in p:
    print(f"\nColumn: {col}, max={m}")
    print("values (first5):", vals[:5])


_IncompleteInputError: incomplete input (293487198.py, line 7)